# WakeWordProject — trening w Google Colab

Ten notebook uruchamia ten sam pipeline co lokalny `train.py`, **bez Dockera** (instalacja pip + klon `openWakeWord` v0.6.0 + konwersja TFLite).

**Zanim uruchomisz:**
- **Runtime → Change runtime type → GPU** (opcjonalnie; przy braku GPU trening i tak działa na CPU).
- **Git:** w następnej komórce ustaw **`REPO_URL`** na **prawdziwy** adres swojego repo (nie zostawiaj `TWOJ_USER`). Przykład: `https://github.com/moj_login/WakeWordProject.git`. Repo musi być **publiczne** albo musisz być zalogowany w Colab z dostępem do prywatnego.
- **Bez GitHuba:** `USE_GIT_CLONE = False`, wgraj ZIP projektu w **Files** → rozpakuj tak, by powstał katalog `/content/WakeWordProject` z podfolderami `colab/` i `scripts/` (np. `!unzip -q WakeWordProject.zip -d /content` jeśli ZIP zawiera jeden folder).
- Sesja Colab może się rozłączyć — warto **Drive** i `OUTPUT_DIR` na dysku.
- Przy **OOM** zmniejsz w YAML `tts_batch_size` / `n_samples`.
- **Błąd `git clone` exit 128:** zły URL, prywatne repo bez dostępu, albo gałąź `BRANCH` nie istnieje (spróbuj `master` zamiast `main`).

In [ ]:
# Opcjonalnie: Google Drive (wyniki w OUTPUT_DIR na dysku)
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess

# --- Ustawienia ---
USE_GIT_CLONE = True
# Wklej TU pełny URL swojego repozytorium (publicznego). NIE zostawiaj TWOJ_USER.
REPO_URL = ""
BRANCH = "main"

PROJECT_ROOT = "/content/WakeWordProject"
OUTPUT_DIR = "/content/wakeword_outputs"
# Przykład na Drive (odkomentuj po zamontowaniu Drive):
# OUTPUT_DIR = "/content/drive/MyDrive/WakeWordProject_outputs"

if USE_GIT_CLONE:
    url = (REPO_URL or "").strip()
    if not url or "TWOJ_USER" in url or "YOUR_" in url.upper():
        raise ValueError(
            "Ustaw REPO_URL na prawdziwy adres HTTPS, np. "
            "https://github.com/twoj_login/WakeWordProject.git\n"
            "Albo: USE_GIT_CLONE = False i wgraj/rozpakuj projekt do "
            f"{PROJECT_ROOT} (foldery colab/, scripts/)."
        )
    if os.path.isdir(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, url, PROJECT_ROOT],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stderr or r.stdout)
        raise RuntimeError(
            f"git clone nie powiódł się (exit {r.returncode}). "
            "Sprawdź URL, dostęp do prywatnego repo oraz czy gałąź "
            f"'{BRANCH}' istnieje (czasem jest 'master' zamiast 'main')."
        ) from None
else:
    assert os.path.isdir(PROJECT_ROOT), (
        f"Brak {PROJECT_ROOT} — wgraj ZIP (Rozpakuj do WakeWordProject) lub włącz USE_GIT_CLONE."
    )

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
%%capture
!apt-get update -qq
!apt-get install -qq -y libespeak-ng-dev espeak-ng git

In [ ]:
%%capture
# Colab często ma nowszy TensorFlow — dla onnx2tf ustawiamy wersję jak w Dockerfile.
!pip uninstall -y tensorflow keras tensorflow-intel tf_keras tensorflow-probability 2>/dev/null || true
!pip install -q -r colab/requirements-colab.txt
!pip install -q --force-reinstall "numpy==1.26.4"

In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT_ROOT)
cmd = [
    sys.executable,
    "colab/colab_train.py",
    "--project_root",
    PROJECT_ROOT,
    "--output_dir",
    OUTPUT_DIR,
    "--training_config",
    "training_configs/hey_lolita_colab.yml",
]
subprocess.run(cmd, check=True)

# Ponowny sam trening (clipy już w OUTPUT_DIR):
# subprocess.run([sys.executable, "colab/colab_train.py", "--project_root", PROJECT_ROOT,
#     "--output_dir", OUTPUT_DIR, "--skip_setup", "--train_only"], check=True)
# Wymuszenie nadpisania: dopisz "--force_overwrite" do listy cmd
# Tylko ONNX: dopisz "--skip_tflite"

## Wyniki

W `OUTPUT_DIR` powinny pojawić się m.in. `hey_lolita.onnx` i `hey_lolita.tflite`. Pobierz je (**Files** w panelu Colab) albo skopiuj na Drive.

Pełniejsza jakość (więcej próbek / kroków): skopiuj parametry z `training_configs/hey_lolita.yml` do pliku colab lub użyj `--training_config training_configs/hey_lolita.yml` (dłużej, większe ryzyko timeoutu i OOM).